# EMS Optimization - Input Modeling for Crash Demand

**Phase 2: Demand Modeling**

This notebook documents the stochastic demand models fitted for the EMS discrete-event simulation.

## Overview

We fit arrival rate models to generate realistic crash events in the simulation:
1. **Homogeneous Poisson Process** - Baseline constant rate model
2. **Non-Homogeneous Poisson Process (NHPP)** - Time-varying rates with hourly and day-of-week factors
3. **Spatial Demand Model** - Precinct-level arrival rates

### Auto-generate missing data
The cell below checks if required processed data exists and generates it automatically if missing.
This ensures each notebook can run independently from a clean state.

In [ ]:
# Auto-generate missing processed data if needed
import sys, os
from pathlib import Path

# Detect project root (works from notebooks/ directory)
_PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if not (_PROJECT_ROOT / 'scripts' / 'generate_all_data.py').exists():
    _PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(_PROJECT_ROOT))
sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

from scripts.generate_all_data import ensure_data
ensure_data(_PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

BASE_DIR = Path('/home/ubuntu/ems-optimization')
PROCESSED = BASE_DIR / 'data/processed'
FIGURES = BASE_DIR / 'results/baseline/figures'

# Load model outputs
hourly_rates = pd.read_csv(PROCESSED / 'demand_lambda_hourly.csv')
dow_factors = pd.read_csv(PROCESSED / 'demand_lambda_dow.csv')
precinct_rates = pd.read_csv(PROCESSED / 'demand_lambda_precinct.csv')

with open(PROCESSED / 'demand_model_summary.json') as f:
    summary = json.load(f)

print(f"Data: {summary['total_crashes']:,} crashes from {summary['date_range']}")

## 1. Homogeneous Poisson Model

The simplest model assumes a constant arrival rate λ.

In [ ]:
lambda_overall = summary['lambda_overall']
print(f"Overall arrival rate: {lambda_overall:.4f} crashes/hour")
print(f"                    = {lambda_overall * 24:.2f} crashes/day")
print(f"")
print(f"Chi-square test for Poisson assumption:")
print(f"  χ² = {summary['chi2_stat']:.2f}, p-value ≈ 0")
print(f"  Result: REJECT homogeneous Poisson (significant time variation)")
print(f"")
print(f"Dispersion (Var/Mean): {summary['dispersion']:.2f}")
print(f"  -> Overdispersion indicates time-varying rates")

## 2. Non-Homogeneous Poisson Process (NHPP)

The NHPP model uses time-varying rates:

$$\lambda(t) = \lambda_{base} \times f_{hour}(h) \times f_{dow}(d)$$

Where:
- $\lambda_{base}$ = 3.48 crashes/hour (overall mean)
- $f_{hour}(h)$ = hourly factor for hour h (0-23)
- $f_{dow}(d)$ = day-of-week factor for day d (0=Monday, 6=Sunday)

In [ ]:
# Display hourly factors
print("Hourly Arrival Rate Factors:")
print(hourly_rates[['hour', 'lambda_per_hour', 'factor']].to_string(index=False))

In [ ]:
# Display day-of-week factors
print("Day-of-Week Factors:")
print(dow_factors[['day_name', 'lambda_per_day', 'factor']].to_string(index=False))

In [ ]:
# Visualize hourly patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(hourly_rates['hour'], hourly_rates['factor'], color='steelblue', alpha=0.7)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Factor (relative to mean)')
ax.set_title('Hourly Demand Pattern')
ax.set_xticks(range(24))

ax = axes[1]
ax.bar(range(7), dow_factors['factor'], color='darkorange', alpha=0.7)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Factor (relative to mean)')
ax.set_title('Day-of-Week Pattern')
ax.set_xticks(range(7))
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.tight_layout()
plt.show()

### Key Findings:
- **Peak hour**: 4-5 PM (factor ≈ 1.52)
- **Low hour**: 5 AM (factor ≈ 0.45)
- **Peak day**: Friday (factor ≈ 1.15)
- **Low day**: Sunday (factor ≈ 0.81)

## 3. Spatial Demand Model (By Precinct)

In [ ]:
print("Precinct-Level Arrival Rates:")
print(precinct_rates.to_string(index=False))
print(f"\nHigh-demand precincts: {summary['high_demand_precincts']}")
print(f"Low-demand precincts: {summary['low_demand_precincts']}")

## 4. CBD vs Non-CBD Comparison

In [ ]:
print(f"CBD accounts for {summary['cbd_pct']:.1f}% of Manhattan crashes")
print(f"")
print(f"Arrival rates:")
print(f"  CBD: {summary['lambda_cbd']:.4f} crashes/hour ({summary['lambda_cbd']*24:.2f}/day)")
print(f"  Non-CBD: {summary['lambda_non_cbd']:.4f} crashes/hour ({summary['lambda_non_cbd']*24:.2f}/day)")

# Compare patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(hourly_rates['hour'], hourly_rates['cbd_factor'], 'o-', label='CBD', linewidth=2)
ax.plot(hourly_rates['hour'], hourly_rates['non_cbd_factor'], 's-', label='Non-CBD', linewidth=2)
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Factor')
ax.set_title('Hourly Patterns: CBD vs Non-CBD')
ax.legend()
ax.set_xticks(range(24))

ax = axes[1]
width = 0.35
x = np.arange(7)
ax.bar(x - width/2, dow_factors['cbd_factor'], width, label='CBD', alpha=0.8)
ax.bar(x + width/2, dow_factors['non_cbd_factor'], width, label='Non-CBD', alpha=0.8)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Factor')
ax.set_title('Day-of-Week Patterns: CBD vs Non-CBD')
ax.set_xticks(x)
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax.legend()

plt.tight_layout()
plt.show()

### CBD vs Non-CBD Insights:
- **CBD peak**: 2 PM (earlier, driven by business activity)
- **Non-CBD peak**: 4 PM (later, more residential/commute patterns)
- **CBD has sharper peaks** during business hours
- **Non-CBD has more weekend activity** relative to weekdays

## 5. Simulation Usage

To generate crash arrivals in the simulation:

```python
def get_arrival_rate(hour, day_of_week, precinct=None):
    """Get λ(t) for given time and optional location."""
    base_rate = 3.48  # crashes/hour overall
    hour_factor = hourly_rates.loc[hour, 'factor']
    dow_factor = dow_factors.loc[day_of_week, 'factor']
    
    if precinct:
        precinct_pct = precinct_rates.loc[precinct, 'pct_of_total'] / 100
        return base_rate * hour_factor * dow_factor * precinct_pct
    
    return base_rate * hour_factor * dow_factor

# Example: Friday at 5 PM
rate = get_arrival_rate(hour=17, day_of_week=4)
print(f"Expected arrival rate: {rate:.2f} crashes/hour")
```

## Output Files

| File | Description |
|------|-------------|
| `demand_lambda_hourly.csv` | 24 hourly arrival rates and factors |
| `demand_lambda_dow.csv` | 7 day-of-week factors |
| `demand_lambda_precinct.csv` | Precinct-level rates |
| `demand_model_summary.json` | Model summary statistics |